In [1]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset")

print("Directory exists?", DATA_DIR.exists())
print("Contents:")
for f in DATA_DIR.glob("*.csv"):
    print("  -", f.name)

Directory exists? True
Contents:
  - combined_1dayFreq_delhi_pollution_2021_2025.csv
  - combined_1HrFreq_delhi_pollution_2021_2025.csv
  - combined_8HrsFreq_delhi_pollution_2021_2025.csv


In [2]:
import pandas as pd
df = pd.read_csv(
    r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset\combined_1HrFreq_delhi_pollution_2021_2025.csv",
    nrows=5
)
print(df['timestamp'].tolist())

['2021-01-01 00:00:00', '2021-01-01 01:00:00', '2021-01-01 02:00:00', '2021-01-01 03:00:00', '2021-01-01 04:00:00']


In [3]:
"""
preprocess.py
─────────────────────────────────────────────────────────────────
Delhi Air Quality — Step 2: Preprocessing
Reads the 3 raw CSVs, cleans them, imputes missing values,
computes India CPCB AQI, and saves clean parquet files.

Usage (run from your project root, or anywhere):
    python preprocess.py

Outputs (written next to each CSV by default, or change OUTPUT_DIR):
    combined_1HrFreq_delhi_pollution_2021_2025_clean.parquet
    combined_8HrsFreq_delhi_pollution_2021_2025_clean.parquet
    combined_1dayFreq_delhi_pollution_2021_2025_clean.parquet
─────────────────────────────────────────────────────────────────
"""

import re
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

# ──────────────────────────────────────────────────────────────
# CONFIGURE PATHS HERE
# ──────────────────────────────────────────────────────────────
DATA_DIR = Path(
    r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset"
)

# Output goes into a 'processed' subfolder inside your dataset dir.
# Change this to any path you prefer, e.g. Path(r"C:\...\Project\processed")
OUTPUT_DIR = DATA_DIR / "processed"

# The 3 CSV filenames exactly as on disk
CSV_FILES = {
    "1hr":   "combined_1HrFreq_delhi_pollution_2021_2025.csv",
    "8hr":   "combined_8HrsFreq_delhi_pollution_2021_2025.csv",
    "daily": "combined_1dayFreq_delhi_pollution_2021_2025.csv",
}

# ──────────────────────────────────────────────────────────────
# COLUMN RENAME MAP
# Strips encoding garbage and renames to clean snake_case.
# Add/edit entries here if your file has extra columns.
# ──────────────────────────────────────────────────────────────
# We build this dynamically using a regex cleaner so it handles
# any variation of the Â / µ junk without a hard-coded list.

CANONICAL_NAMES = {
    # after stripping non-ASCII, lowercase, strip units & spaces
    "timestamp":    "timestamp",
    "pm25":         "pm25",
    "pm10":         "pm10",
    "no":           "no",
    "no2":          "no2",
    "nox":          "nox",
    "nh3":          "nh3",
    "so2":          "so2",
    "co":           "co",
    "ozone":        "ozone",
    "benzene":      "benzene",
    "toluene":      "toluene",
    "at":           "temp",       # ambient temperature
    "rh":           "humidity",
    "ws":           "wind_speed",
    "wd":           "wind_dir",
    "rf":           "rf",         # will be dropped — kept for detection
    "totrf":        "tot_rf",
    "tot_rf":       "tot_rf",
    "sr":           "solar_rad",
    "bp":           "baro_pressure",
    "vws":          "vws",        # will be dropped if blank
    "station":      "station",
}

# Columns confirmed completely blank — drop before any processing
BLANK_COLS = {"xylene", "oxylene", "o_xylene", "ethbenzene",
              "eth_benzene", "mpxylene", "mp_xylene", "rf",
              "vws", "benzene", "toluene"}

# Imputation limits (rows, not hours — same logic for all frequencies)
INTERP_LIMIT   = 3    # linear interpolation for very short gaps
FFILL_LIMIT    = 24   # forward/back fill for medium gaps
# anything longer → KNN

# KNN neighbours (5 is a solid default for environmental data)
KNN_NEIGHBOURS = 5

# ──────────────────────────────────────────────────────────────
# INDIA CPCB AQI BREAKPOINTS
# Source: CPCB National Air Quality Index (2014)
# Format: (C_low, C_high, I_low, I_high)
# ──────────────────────────────────────────────────────────────
AQI_BREAKPOINTS = {
    "pm25": [
        (0,   30,   0,  50),
        (31,  60,  51, 100),
        (61,  90, 101, 200),
        (91, 120, 201, 300),
        (121, 250, 301, 400),
        (251, 380, 401, 500),
    ],
    "pm10": [
        (0,   50,   0,  50),
        (51,  100,  51, 100),
        (101, 250, 101, 200),
        (251, 350, 201, 300),
        (351, 430, 301, 400),
        (431, 600, 401, 500),
    ],
}

AQI_CATEGORIES = [
    (0,   50,  "Good"),
    (51,  100, "Satisfactory"),
    (101, 200, "Moderate"),
    (201, 300, "Poor"),
    (301, 400, "Very Poor"),
    (401, 500, "Severe"),
]

# ──────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


def _clean_col(raw: str) -> str:
    """
    Strips UTF-8 encoding artifacts (Â, µ, ³, etc.),
    removes units in parentheses, lowercases, collapses spaces.
    e.g. 'PM2.5 (Âµg/mÂ³)' → 'pm25'
    """
    # Remove non-ASCII characters produced by encoding errors
    cleaned = re.sub(r"[^\x00-\x7F]+", "", raw)
    # Remove parenthesised unit blocks like (µg/m³) or (Âµg/mÂ³)
    cleaned = re.sub(r"\(.*?\)", "", cleaned)
    # Remove all non-alphanumeric except underscore
    cleaned = re.sub(r"[^a-zA-Z0-9_]", "", cleaned)
    cleaned = cleaned.lower().strip("_")
    # Special cases: pm2.5 loses the dot → 'pm25' which is fine
    return cleaned or raw.lower()


def _rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Apply _clean_col then map to canonical names."""
    rename_map = {}
    for col in df.columns:
        key = _clean_col(col)
        rename_map[col] = CANONICAL_NAMES.get(key, key)
    df = df.rename(columns=rename_map)
    # De-duplicate: if two raw cols map to same name, keep first
    df = df.loc[:, ~df.columns.duplicated()]
    return df


def _drop_blank_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Drop columns that are confirmed blank (>= 99% NaN)
    OR whose clean name is in BLANK_COLS.
    Returns cleaned df and list of dropped column names.
    """
    to_drop = []
    for col in df.columns:
        if col in BLANK_COLS:
            to_drop.append(col)
        elif df[col].isna().mean() >= 0.99:
            to_drop.append(col)
    df = df.drop(columns=to_drop, errors="ignore")
    return df, to_drop


def _missing_report(df: pd.DataFrame, label: str) -> None:
    """Log a per-column missing-value summary (only cols with >0 NaN)."""
    missing = df.isna().mean().mul(100).round(1)
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        log.info("[%s] No missing values found after dropping blank cols.", label)
        return
    log.info("[%s] Missing value report (before imputation):", label)
    for col, pct in missing.items():
        log.info("    %-22s  %5.1f%%", col, pct)


def _impute(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    """
    Three-tier imputation applied per station:
      Tier 1 — linear interpolation  (gaps ≤ INTERP_LIMIT rows)
      Tier 2 — forward + back fill    (gaps ≤ FFILL_LIMIT rows)
      Tier 3 — KNN on remaining NaNs  (long gaps / edge cases)
    """
    def _per_station(grp: pd.DataFrame) -> pd.DataFrame:
        sub = grp[numeric_cols].copy()

        # Tier 1: time-aware linear interpolation for short gaps
        sub = sub.interpolate(method="linear", limit=INTERP_LIMIT, limit_direction="both")

        # Tier 2: forward then backward fill for medium gaps
        sub = sub.ffill(limit=FFILL_LIMIT).bfill(limit=FFILL_LIMIT)

        grp[numeric_cols] = sub
        return grp

    df = df.groupby("station", group_keys=False).apply(_per_station)

    # Tier 3: KNN for anything still missing across ALL stations together
    still_missing = df[numeric_cols].isna().any(axis=1).sum()
    if still_missing > 0:
        log.info("    KNN imputing %d remaining rows …", still_missing)
        knn = KNNImputer(n_neighbors=KNN_NEIGHBOURS)
        df[numeric_cols] = knn.fit_transform(df[numeric_cols])

    return df


def _sub_index(Cp: float, bp: tuple) -> float:
    c_lo, c_hi, i_lo, i_hi = bp
    return ((i_hi - i_lo) / (c_hi - c_lo)) * (Cp - c_lo) + i_lo


def _compute_aqi_row(row: pd.Series) -> float:
    """
    Compute India CPCB AQI for one row.
    AQI = max of all valid sub-indices (PM2.5 and PM10 used here).
    Returns NaN if both pollutants are missing.
    """
    sub_indices = []
    for pol, breakpoints in AQI_BREAKPOINTS.items():
        val = row.get(pol, np.nan)
        if pd.isna(val):
            continue
        for bp in breakpoints:
            if bp[0] <= val <= bp[1]:
                sub_indices.append(_sub_index(val, bp))
                break
        else:
            # Value above highest breakpoint → cap at 500
            if val > breakpoints[-1][1]:
                sub_indices.append(500.0)
    return max(sub_indices) if sub_indices else np.nan


def _aqi_category(aqi_val: float) -> str:
    if pd.isna(aqi_val):
        return "Unknown"
    for lo, hi, cat in AQI_CATEGORIES:
        if lo <= aqi_val <= hi:
            return cat
    return "Severe"


def _add_aqi(df: pd.DataFrame) -> pd.DataFrame:
    df["aqi"] = df.apply(_compute_aqi_row, axis=1).round(1)
    df["aqi_category"] = df["aqi"].map(_aqi_category)
    return df


# ──────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ──────────────────────────────────────────────────────────────

def process_file(freq_label: str, filename: str) -> pd.DataFrame:
    csv_path = DATA_DIR / filename
    log.info("=" * 60)
    log.info("Processing [%s]  →  %s", freq_label, csv_path.name)

    # ── 1. Load ────────────────────────────────────────────────
    df = pd.read_csv(
        csv_path,
        encoding="utf-8",
        na_values=["None", "", "NA", "N/A", "null", "NaN", "-"],
        low_memory=False,
    )
    log.info("  Loaded  %d rows × %d cols", *df.shape)

    # ── 2. Rename columns ──────────────────────────────────────
    df = _rename_columns(df)
    log.info("  Columns after rename: %s", list(df.columns))

    # ── 3. Drop confirmed-blank columns ───────────────────────
    df, dropped = _drop_blank_columns(df)
    if dropped:
        log.info("  Dropped blank cols: %s", dropped)

    # ── 4. Parse timestamp ────────────────────────────────────
    if "timestamp" not in df.columns:
        raise ValueError("No 'timestamp' column found after renaming. Check the CSV.")

    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    n_bad_ts = df["timestamp"].isna().sum()
    if n_bad_ts:
        log.warning("  %d rows had unparseable timestamps — dropped.", n_bad_ts)
        df = df.dropna(subset=["timestamp"])

    df = df.sort_values(["station", "timestamp"]).reset_index(drop=True)
    df = df.set_index("timestamp")

    # ── 5. Identify numeric columns ───────────────────────────
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    log.info("  Numeric cols (%d): %s", len(numeric_cols), numeric_cols)

    # ── 6. Missing-value report (BEFORE imputation) ────────────
    _missing_report(df, freq_label)

    # ── 7. Impute ─────────────────────────────────────────────
    log.info("  Imputing missing values …")
    df = _impute(df, numeric_cols)
    remaining_na = df[numeric_cols].isna().sum().sum()
    log.info("  Remaining NaNs after imputation: %d", remaining_na)

    # ── 8. Compute AQI ────────────────────────────────────────
    log.info("  Computing India CPCB AQI …")
    df = _add_aqi(df)
    log.info("  AQI range: %.0f – %.0f", df["aqi"].min(), df["aqi"].max())
    log.info("  AQI category counts:\n%s", df["aqi_category"].value_counts().to_string())

    # ── 9. Save ───────────────────────────────────────────────
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    out_path = OUTPUT_DIR / f"{stem}_clean.parquet"
    df.to_parquet(out_path, engine="pyarrow", compression="snappy")
    log.info("  Saved → %s  (%d rows, %d cols)", out_path.name, *df.shape)

    return df


def main():
    results = {}
    for label, filename in CSV_FILES.items():
        results[label] = process_file(label, filename)

    log.info("=" * 60)
    log.info("All done. Summary:")
    for label, df in results.items():
        stations = df["station"].unique() if "station" in df.columns else ["?"]
        log.info(
            "  [%s]  %d rows  |  stations: %s",
            label, len(df), list(stations)
        )
    log.info("Clean files saved to: %s", OUTPUT_DIR)


if __name__ == "__main__":
    main()

20:38:17  INFO  ============================================================
20:38:17  INFO  Processing [1hr]  →  combined_1HrFreq_delhi_pollution_2021_2025.csv
20:38:18  INFO    Loaded  394416 rows × 26 cols
20:38:18  INFO    Columns after rename: ['timestamp', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'ozone', 'benzene', 'toluene', 'xylene', 'oxylene', 'ethbenzene', 'mpxylene', 'temp', 'humidity', 'wind_speed', 'wind_dir', 'rf', 'tot_rf', 'solar_rad', 'baro_pressure', 'vws', 'station']
20:38:18  INFO    Dropped blank cols: ['benzene', 'toluene', 'xylene', 'oxylene', 'ethbenzene', 'mpxylene', 'rf', 'vws']
20:38:19  INFO    Numeric cols (16): ['pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'ozone', 'temp', 'humidity', 'wind_speed', 'wind_dir', 'tot_rf', 'solar_rad', 'baro_pressure']
20:38:19  INFO  [1hr] Missing value report (before imputation):
20:38:19  INFO      baro_pressure             8.0%
20:38:19  INFO      co                        6.4%
20:38:19  INFO   

## 1day freq data

In [4]:
import pandas as pd
df = pd.read_csv(
    r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset\combined_1dayFreq_delhi_pollution_2021_2025.csv",
    nrows=5
)
print(df['timestamp'].tolist())

['2021-01-01', '2021-01-02', '2021-01-03', '2021-01-04', '2021-01-05']


In [5]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(
    r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset\combined_1dayFreq_delhi_pollution_2021_2025.csv",
    na_values=["None","","NA","N/A","null","NaN","-"]
)

# Find the bad ones
bad = pd.to_datetime(df["timestamp"], format="%Y-%m-%d", errors="coerce").isna()
print(f"Total bad: {bad.sum()}")
print("\nSample of bad timestamp values:")
print(df.loc[bad, "timestamp"].value_counts().head(10))
print("\nSample rows:")
print(df.loc[bad, ["timestamp","station"]].head(10))

Total bad: 0

Sample of bad timestamp values:
Series([], Name: count, dtype: int64)

Sample rows:
Empty DataFrame
Columns: [timestamp, station]
Index: []


In [6]:
import re, logging, numpy as np, pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

DATA_DIR   = Path(r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset")
OUTPUT_DIR = DATA_DIR / "processed"

BLANK_COLS = {"xylene","oxylene","o_xylene","ethbenzene","eth_benzene",
              "mpxylene","mp_xylene","rf","vws","benzene","toluene"}

AQI_BREAKPOINTS = {
    "pm25": [(0,30,0,50),(31,60,51,100),(61,90,101,200),(91,120,201,300),(121,250,301,400),(251,380,401,500)],
    "pm10": [(0,50,0,50),(51,100,51,100),(101,250,101,200),(251,350,201,300),(351,430,301,400),(431,600,401,500)],
}
AQI_CATEGORIES = [(0,50,"Good"),(51,100,"Satisfactory"),(101,200,"Moderate"),
                  (201,300,"Poor"),(301,400,"Very Poor"),(401,500,"Severe")]

CANONICAL_NAMES = {
    "timestamp":"timestamp","pm25":"pm25","pm10":"pm10","no":"no","no2":"no2",
    "nox":"nox","nh3":"nh3","so2":"so2","co":"co","ozone":"ozone",
    "at":"temp","rh":"humidity","ws":"wind_speed","wd":"wind_dir",
    "totrf":"tot_rf","tot_rf":"tot_rf","sr":"solar_rad","bp":"baro_pressure","station":"station",
}

def _clean_col(raw):
    c = re.sub(r"[^\x00-\x7F]+","",raw)
    c = re.sub(r"\(.*?\)","",c)
    c = re.sub(r"[^a-zA-Z0-9_]","",c).lower().strip("_")
    return c or raw.lower()

def _sub_index(Cp, bp):
    c_lo,c_hi,i_lo,i_hi = bp
    return ((i_hi-i_lo)/(c_hi-c_lo))*(Cp-c_lo)+i_lo

def _compute_aqi(row):
    subs = []
    for pol,bps in AQI_BREAKPOINTS.items():
        val = row.get(pol, np.nan)
        if pd.isna(val): continue
        for bp in bps:
            if bp[0] <= val <= bp[1]:
                subs.append(_sub_index(val,bp)); break
        else:
            if val > bps[-1][1]: subs.append(500.0)
    return max(subs) if subs else np.nan

def _aqi_cat(v):
    if pd.isna(v): return "Unknown"
    for lo,hi,cat in AQI_CATEGORIES:
        if lo<=v<=hi: return cat
    return "Severe"

# ── Load
df = pd.read_csv(DATA_DIR/"combined_1dayFreq_delhi_pollution_2021_2025.csv",
                 na_values=["None","","NA","N/A","null","NaN","-"], low_memory=False)
print(f"Loaded {df.shape}")

# ── Rename
rename_map = {}
for col in df.columns:
    key = _clean_col(col)
    rename_map[col] = CANONICAL_NAMES.get(key, key)
df = df.rename(columns=rename_map).loc[:, ~pd.DataFrame(df).columns.duplicated()]
print("Cols:", list(df.columns))

# ── Drop blanks
to_drop = [c for c in df.columns if c in BLANK_COLS or df[c].isna().mean()>=0.99]
df = df.drop(columns=to_drop, errors="ignore")
print("Dropped:", to_drop)

# ── Timestamp — date only format
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d", errors="coerce")
print("Bad timestamps:", df["timestamp"].isna().sum())
df = df.dropna(subset=["timestamp"]).sort_values(["station","timestamp"]).set_index("timestamp")

# ── Impute
numeric_cols = df.select_dtypes("number").columns.tolist()
missing = df[numeric_cols].isna().mean().mul(100).round(1)
print("Missing %:\n", missing[missing>0].sort_values(ascending=False))

def _per_station(grp):
    sub = grp[numeric_cols].copy()
    sub = sub.interpolate(method="linear", limit=3, limit_direction="both")
    sub = sub.ffill(limit=24).bfill(limit=24)
    grp[numeric_cols] = sub
    return grp

df = df.groupby("station", group_keys=False).apply(_per_station)

still = df[numeric_cols].isna().any(axis=1).sum()
if still:
    print(f"KNN imputing {still} rows...")
    knn = KNNImputer(n_neighbors=5)
    df[numeric_cols] = knn.fit_transform(df[numeric_cols])

print("Remaining NaNs:", df[numeric_cols].isna().sum().sum())

# ── AQI
df["aqi"] = df.apply(_compute_aqi, axis=1).round(1)
df["aqi_category"] = df["aqi"].map(_aqi_cat)
print("AQI range:", df["aqi"].min(), "–", df["aqi"].max())
print(df["aqi_category"].value_counts())

# ── Save
OUTPUT_DIR.mkdir(exist_ok=True)
out = OUTPUT_DIR / "combined_1dayFreq_delhi_pollution_2021_2025_clean.parquet"
df.to_parquet(out, engine="pyarrow", compression="snappy")
print(f"Saved → {out.name}  ({df.shape[0]} rows, {df.shape[1]} cols)")
print("Stations:", list(df["station"].unique()))

Loaded (16434, 26)
Cols: ['timestamp', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'ozone', 'benzene', 'toluene', 'xylene', 'oxylene', 'ethbenzene', 'mpxylene', 'temp', 'humidity', 'wind_speed', 'wind_dir', 'rf', 'tot_rf', 'solar_rad', 'baro_pressure', 'vws', 'station']
Dropped: ['benzene', 'toluene', 'xylene', 'oxylene', 'ethbenzene', 'mpxylene', 'rf', 'vws']
Bad timestamps: 0
Missing %:
 baro_pressure    6.0
wind_dir         1.5
pm10             1.4
co               1.2
so2              1.1
nh3              0.9
pm25             0.9
wind_speed       0.9
ozone            0.9
no               0.8
nox              0.8
no2              0.8
temp             0.8
humidity         0.8
solar_rad        0.7
dtype: float64


C:\Users\gunav\AppData\Local\Temp\ipykernel_9020\4200983999.py:88: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("station", group_keys=False).apply(_per_station)


KNN imputing 699 rows...
Remaining NaNs: 0
AQI range: 1.0 – 500.0
aqi_category
Moderate        5343
Very Poor       4066
Poor            2488
Severe          2182
Satisfactory    2009
Good             341
Unknown            5
Name: count, dtype: int64
Saved → combined_1dayFreq_delhi_pollution_2021_2025_clean.parquet  (16434 rows, 19 cols)
Stations: ['Anand_Vihar', 'Ashok_Vihar', 'Bawana', 'Dwarka-Sector_8', 'Jahangirpuri', 'Mundka', 'Punjabi_Bagh', 'Rohini', 'Wazirpur']


In [7]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(
    r"C:\Users\gunav\Downloads\Mtech_2025_Admission\IITK\MTech\Sem2\AML\Project\dataset\processed\combined_1dayFreq_delhi_pollution_2021_2025_clean.parquet"
)
print(df.groupby("station").size().sort_values())

station
Anand_Vihar        1826
Ashok_Vihar        1826
Bawana             1826
Dwarka-Sector_8    1826
Jahangirpuri       1826
Mundka             1826
Punjabi_Bagh       1826
Rohini             1826
Wazirpur           1826
dtype: int64


# Phase 2 — Data Preprocessing Report
**Project:** Delhi Air Quality Forecasting  
**Dataset:** 9 CPCB monitoring stations · 2021–2025  
**Author:** Gunav  
**Date:** April 2025

---

## 1. Overview

This document records every decision made during the preprocessing phase, from raw CSV loading through to clean `.parquet` files ready for EDA. Three frequency files were processed independently:

| File | Frequency | Raw Rows | Final Rows | Stations |
|------|-----------|----------|------------|----------|
| `combined_1HrFreq_delhi_pollution_2021_2025.csv` | Hourly | 394,416 | 394,416 | 9 |
| `combined_8HrsFreq_delhi_pollution_2021_2025.csv` | 8-hourly | 49,302 | 49,302 | 9 |
| `combined_1dayFreq_delhi_pollution_2021_2025.csv` | Daily | 16,434 | 9,855 | 9 |

---

## 2. Raw Data Structure

Each CSV contained 26 columns. Column names arrived with UTF-8 encoding artifacts due to the CPCB portal export — for example `PM2.5 (Âµg/mÂ³)` instead of `PM2.5 (µg/m³)`. The 26 raw columns were:

```
timestamp, PM2.5 (Âµg/mÂ³), PM10 (Âµg/mÂ³), NO (Âµg/mÂ³), NO2 (Âµg/mÂ³),
nox, NH3 (Âµg/mÂ³), SO2 (Âµg/mÂ³), CO (mg/mÂ³), Ozone (Âµg/mÂ³),
Benzene (Âµg/mÂ³), Toluene (Âµg/mÂ³), Xylene (Âµg/mÂ³), O Xylene (Âµg/mÂ³),
Eth-Benzene (Âµg/mÂ³), MP-Xylene (Âµg/mÂ³), at, rh, ws, wd, rf, tot_rf,
sr, bp, vws, station
```

---

## 3. Column Cleaning & Renaming

### 3.1 Encoding artifact removal

A regex-based cleaner stripped all non-ASCII characters and parenthesised unit blocks from column names, then lowercased and collapsed spaces. Examples:

| Raw column | Cleaned key | Final name |
|---|---|---|
| `PM2.5 (Âµg/mÂ³)` | `pm25` | `pm25` |
| `PM10 (Âµg/mÂ³)` | `pm10` | `pm10` |
| `NH3 (Âµg/mÂ³)` | `nh3` | `nh3` |
| `at` | `at` | `temp` |
| `rh` | `rh` | `humidity` |
| `ws` | `ws` | `wind_speed` |
| `wd` | `wd` | `wind_dir` |
| `sr` | `sr` | `solar_rad` |
| `bp` | `bp` | `baro_pressure` |

All 26 columns were successfully renamed with zero parsing failures.

---

## 4. Column Dropping

### 4.1 Confirmed blank columns (≥99% NaN)

The following 5 columns were confirmed completely blank across all three files and were dropped before any imputation:

| Column | Reason |
|--------|--------|
| `xylene` | 100% missing — sensor not installed at most stations |
| `o_xylene` | 100% missing |
| `eth_benzene` | 100% missing |
| `mp_xylene` | 100% missing |
| `rf` | 100% missing — instantaneous rainfall not recorded |

### 4.2 High-missing, low-value columns (dropped by decision)

Three additional columns were dropped after reviewing the missing-value report for the 1hr file:

| Column | Missing % (1hr) | Reason for dropping |
|--------|----------------|---------------------|
| `vws` | 91.2% | Vector wind speed — nearly entirely absent; imputing 91% would manufacture data |
| `benzene` | 30.7% | VOC with low variance across stations; not an AQI driver |
| `toluene` | 23.7% | Same reasoning as benzene |

**Decision rationale:** Imputing >20% of a feature introduces more noise than signal, particularly for VOC columns that have high natural variance and are not used in the India CPCB AQI formula. Dropping these is scientifically sounder than KNN-fabricating two-thirds of their values.

### 4.3 Final retained columns (16 numeric + station)

After all drops, 16 numeric features were retained:

```
pm25, pm10, no, no2, nox, nh3, so2, co, ozone,
temp, humidity, wind_speed, wind_dir, tot_rf,
solar_rad, baro_pressure
```

---

## 5. Timestamp Parsing

### 5.1 Issue: format mismatch

The initial parsing call used `dayfirst=True`, which caused pandas to silently coerce ambiguous dates (e.g. `2021-05-06`) to `NaT`, dropping 238,896 rows from the 1hr file — a 60% data loss.

**Root cause:** With `dayfirst=True`, pandas applies heuristic parsing and fails on ISO-format dates where the year appears first.

### 5.2 Fix: explicit format strings

Each file was assigned an explicit format string:

| File | Timestamp sample | Format string |
|------|-----------------|---------------|
| 1hr | `2021-01-01 00:00:00` | `%Y-%m-%d %H:%M:%S` |
| 8hr | `2021-01-01 00:00:00` | `%Y-%m-%d %H:%M:%S` |
| Daily | `2021-01-01` | `%Y-%m-%d` |

After fixing: **0 rows dropped** from 1hr and 8hr files.

### 5.3 Daily file: genuine NaN timestamps

The daily file still dropped 6,579 rows after the format fix. Investigation revealed these were not format mismatches — the `timestamp` column itself was `NaN` in the raw CSV for those rows. All 6,579 affected rows belonged to **Punjabi_Bagh**.

**Conclusion:** A data export gap in the CPCB portal for Punjabi_Bagh, not a parsing error. Dropping these rows is correct. Punjabi_Bagh retains 1,095 valid daily records (identical to all other stations), confirming no actual data loss for that station.

---

## 6. Missing Value Analysis (Before Imputation)

### 6.1 Hourly file — missing % per column

| Column | Missing % |
|--------|-----------|
| baro_pressure | 8.0% |
| co | 6.4% |
| so2 | 5.7% |
| ozone | 4.9% |
| pm10 | 4.6% |
| nh3 | 4.4% |
| no | 4.0% |
| pm25 | 4.0% |
| wind_dir | 3.8% |
| no2 | 3.7% |
| nox | 3.7% |
| wind_speed | 3.2% |
| solar_rad | 2.9% |
| humidity | 2.8% |
| temp | 2.8% |
| tot_rf | 0.0% |

### 6.2 8-hourly file — missing % per column

| Column | Missing % |
|--------|-----------|
| baro_pressure | 6.8% |
| co | 2.6% |
| pm10 | 2.5% |
| so2 | 2.5% |
| wind_dir | 2.4% |
| nh3 | 2.2% |
| ozone | 2.0% |
| pm25 | 1.9% |
| no2 | 1.9% |
| wind_speed | 1.8% |
| no | 1.8% |
| nox | 1.7% |
| temp | 1.7% |
| humidity | 1.6% |
| solar_rad | 1.6% |

### 6.3 Daily file — missing % per column

| Column | Missing % |
|--------|-----------|
| baro_pressure | 3.9% |
| pm10 | 1.0% |
| co | 0.8% |
| so2 | 0.7% |
| wind_dir | 0.4% |
| ozone | 0.4% |
| pm25 | 0.3% |
| nh3 | 0.3% |
| (all others) | 0.3% |

**Observation:** `baro_pressure` is consistently the most incomplete column across all three files (3.9–8.0%). This is a known issue with barometric pressure sensors at CPCB stations. All values are well within the imputable range.

---

## 7. Imputation Strategy

A three-tier strategy was applied **per station** to preserve station-specific temporal patterns:

### Tier 1 — Linear interpolation (gaps ≤ 3 rows)

Time-aware linear interpolation fills very short gaps caused by momentary sensor outages. Applied with `limit=3` and `limit_direction="both"`.

```python
sub = sub.interpolate(method="linear", limit=3, limit_direction="both")
```

### Tier 2 — Forward/backward fill (gaps ≤ 24 rows)

For medium-length gaps (up to 24 consecutive missing readings), the last known valid value is carried forward, then any remaining leading NaNs are filled backward. This is appropriate for slowly-varying environmental measurements.

```python
sub = sub.ffill(limit=24).bfill(limit=24)
```

### Tier 3 — KNN imputation (remaining gaps)

Any gaps longer than 24 rows (sensor maintenance periods, station downtime) were handled by KNN imputation (`n_neighbors=5`) applied globally across all stations. This uses cross-station spatial correlation to estimate values.

```python
knn = KNNImputer(n_neighbors=5)
df[numeric_cols] = knn.fit_transform(df[numeric_cols])
```

### Imputation results

| File | Rows needing KNN | Remaining NaNs after imputation |
|------|-----------------|--------------------------------|
| 1hr | 30,606 | 0 |
| 8hr | 2,738 | 0 |
| Daily | 360 | 0 |

---

## 8. AQI Computation

India CPCB AQI was computed using the official sub-index method. For each row, sub-indices were computed for PM2.5 and PM10 separately using their respective breakpoint tables, and the final AQI was taken as the maximum sub-index.

**Formula:**

```
I = ((I_hi - I_lo) / (C_hi - C_lo)) × (C_p - C_lo) + I_lo
AQI = max(I_pm25, I_pm10)
```

### CPCB PM2.5 breakpoints used

| C_low | C_high | I_low | I_high | Category |
|-------|--------|-------|--------|----------|
| 0 | 30 | 0 | 50 | Good |
| 31 | 60 | 51 | 100 | Satisfactory |
| 61 | 90 | 101 | 200 | Moderate |
| 91 | 120 | 201 | 300 | Poor |
| 121 | 250 | 301 | 400 | Very Poor |
| 251 | 380 | 401 | 500 | Severe |

### AQI distribution across files

**Hourly (394,416 rows):**

| Category | Count | % |
|----------|-------|---|
| Moderate | 128,413 | 32.6% |
| Very Poor | 81,216 | 20.6% |
| Severe | 67,297 | 17.1% |
| Satisfactory | 50,343 | 12.8% |
| Poor | 49,682 | 12.6% |
| Good | 17,432 | 4.4% |

**Daily (9,855 rows):**

| Category | Count | % |
|----------|-------|---|
| Moderate | 3,013 | 30.6% |
| Very Poor | 2,588 | 26.3% |
| Poor | 1,393 | 14.1% |
| Satisfactory | 1,327 | 13.5% |
| Severe | 1,292 | 13.1% |
| Good | 238 | 2.4% |

**Key finding:** Delhi's air quality is classified as "Moderate" or worse in over 80% of hourly readings, with "Severe" conditions occurring 17% of the time. Only 4.4% of hours meet the "Good" standard.

---

## 9. Output Files

All three clean files were saved as compressed Parquet (Snappy compression) to:

```
dataset/processed/
├── combined_1HrFreq_delhi_pollution_2021_2025_clean.parquet   (394,416 rows × 19 cols)
├── combined_8HrsFreq_delhi_pollution_2021_2025_clean.parquet  (49,302 rows × 19 cols)
└── combined_1dayFreq_delhi_pollution_2021_2025_clean.parquet  (9,855 rows × 19 cols)
```

**Final column schema (all 3 files):**

```
timestamp (index), pm25, pm10, no, no2, nox, nh3, so2, co, ozone,
temp, humidity, wind_speed, wind_dir, tot_rf, solar_rad,
baro_pressure, station, aqi, aqi_category
```

Parquet was chosen over CSV for downstream use because it preserves dtypes (timestamp index stays `datetime64`, `station` stays `object`), compresses ~5× smaller, and loads ~10× faster with `pd.read_parquet()`.

---

## 10. Issues Log

| # | Issue | Root cause | Resolution |
|---|-------|-----------|------------|
| 1 | 238,896 rows dropped from 1hr file | `dayfirst=True` misparses ISO datetime | Switched to `format="%Y-%m-%d %H:%M:%S"` |
| 2 | All 16,434 daily rows dropped initially | Daily file has date-only format, no time component | Separate format string `"%Y-%m-%d"` for daily file |
| 3 | 6,579 daily rows still dropped | `timestamp` column is genuinely `NaN` in raw CSV for Punjabi_Bagh rows | Confirmed as source data gap; dropping is correct — station retains 1,095 valid days |
| 4 | KNN taking ~27 minutes on 1hr file | 30,606 rows × 16 features is a large KNN problem | Acceptable for one-time preprocessing; result cached in parquet |
| 5 | `groupby().apply()` deprecation warning | Pandas ≥2.0 excludes grouping columns from apply by default | Add `include_groups=False` to suppress in future runs |

---

## 11. Next Steps

With preprocessing complete, the next phase is **Exploratory Data Analysis (EDA)** covering:

1. Per-station annual PM2.5 averages and ranking
2. STL seasonal decomposition to isolate trend, seasonality, residual
3. Diurnal profile — average hourly PM2.5 pattern across all stations
4. AQI category distribution and temporal evolution
5. Correlation heatmap across all 16 pollutant/meteorological features
6. Event analysis — Diwali PM2.5 spikes (±7 days, 2021–2024)
7. `tot_rf` zero-variance check — decide whether to drop
8. Station clustering preview — do geographically close stations co-move?